# 单晶铜热膨胀行为及线膨胀系数的分子动力学研究

小组成员：【请填写姓名及实际分工】

计算材料学课程项目。本文由原始计算数据生成；研究定位为方法复现与势函数适用性评价。

## 摘要

采用LAMMPS和Mishin等人2001年铜EAM1势，对三向周期性FCC铜进行近零外压NPT分子动力学模拟，研究300–800 K的晶格常数与体积。设置6个温度、每点3次独立重复，并在温区端点检查尺寸、时间步与时长。实验参考采用NIST SRM 736的1990修订表，统一以300 K长度为参考。

**当前为论文方法稿：正式模拟尚未全部完成，结果、摘要数值和最终结论暂不填写。现有试跑仅用于运行检查，不作为18条正式轨迹的替代。**

## 1. 研究背景与文献比较

铜受热后的平均原子位置间距增加，反映势能的非谐性。谐振动模型本身不能给出普通热膨胀。NPT允许体积随温度调整，适合获得近零外压的平衡晶格参数。

Alper与Politzer（1999）以及Kazanc等（2006）展示经验势有限温度研究；Narasimhan与de Gironcoli（2002）、Rudin等（2002）采用电子结构与准谐自由能路线。Lysogorskiy等（2021）直接比较铜的多类势；Olsson等（2023）用Mishin势研究单晶及晶界影响。它们支持本项目路线，但不能据此宣称我们已获得相同精度。

本项目保留原选题的Mishin EAM1，不额外开展机器学习势或DFT计算。详细的25项来源、阅读证据和方法对比见 [文献调研](docs/literature_review.md)。

## 2. 模型与计算方法

FCC常规晶胞沿[100]、[010]、[001]排列，10×10×10晶胞共4000原子，三向周期性边界，无自由表面和晶界。初始a=3.615 Å，先做原子和晶胞能量最小化。Cu质量63.546 u。势文件为NIST `Cu01.eam.alloy`，调用`eam/alloy`；来源与哈希见potentials/SOURCE.md。

使用metal单位制：长度Å、时间ps、能量eV、压力bar。NPT设温300、400、500、600、700、800 K，外压0 bar，等比例调整盒长；时间步0.001 ps（1 fs），温度阻尼0.1 ps，压力阻尼1 ps。每点200 ps平衡、500 ps采样，间隔0.1 ps输出未作长时间平均的标量，结构每100 ps输出。

3个重复的种子基数为13579、24681、97531，实际种子=基数+19×温度，每个温度独立初始化。端点各增加12³晶胞、0.5 fs步长和1 ns平衡+1 ns采样三项检查，总计24条轨迹。所有具体参数见run_study.py及outputs/study_manifest.json。

### 2.1 数据处理

每帧由a=Lx/n取得晶格常数，再对平衡期之后的样本求均值。立方盒保持等比例变化，体积同时从LAMMPS直接读取。主结果定义为

$$\bar\alpha_{300-800}=\frac{\bar a(800)-\bar a(300)}{500\bar a(300)}.$$

这是工程温区平均，区别于瞬时系数$\alpha(T)=a(T)^{-1}da/dT$。不把体膨胀系数直接当作线膨胀系数，也不把局域平均键长等同于晶格常数。

每条轨迹比较前后半段，采用1、2、5、10、20、50 ps分块检查时间相关性，以20 ps分块报告单轨迹均值标准误。每温度汇总3条独立轨迹；系数的不确定度以3个重复端点系数的标准误和t分布（自由度2）得到。重复数较少，统计区间需谨慎解释。

端点验证与基准r1比较，预设平均系数变化±2%为课程项目的实用目标，不能视作普适收敛标准。单次检查不足以精确量化所有系统误差。

### 2.2 实验参考与公平比较

采用[NIST历史修订证书](https://tsapps.nist.gov/srmext/certificates/archives/736.pdf)第1页表值，原表参考293 K，换算为$\epsilon_{300}(T)=[1+e_{293}(T)]/[1+e_{293}(300)]-1$。同定义300–800 K参考平均约18.604×10⁻⁶ K⁻¹。原始表与转换说明见[data/reference/SOURCE.md](data/reference/SOURCE.md)。实验为高纯铜试样，MD为理想周期晶体；两者差别属于讨论内容。

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image
root=Path.cwd()
assert (root/'config.json').exists(), '请在项目根目录打开Notebook'
display(pd.read_csv('data/reference/nist_srm736.csv'))

,temperature_K,expansion_from_293_ppm,expansivity_fixed293_micro_Kinv
0,300,117,16.71
1,400,1831,17.53
2,500,3620,18.25
3,600,5480,18.95
4,700,7411,19.69
5,800,9420,20.51


## 3. 计算结果与可靠性检查

In [2]:
from run_study import manifest, folder
records=[]
for j in manifest():
    p=folder(j)/'metadata.json'
    state=json.loads(p.read_text(encoding='utf8')).get('status') if p.exists() else 'pending'
    records.append({'任务':j['id'],'状态':state,'温度/K':j['temperature']})
display(pd.DataFrame(records))

,任务,状态,温度/K
0,prod_T300_r1,pending,300
1,prod_T400_r1,pending,400
2,prod_T500_r1,pending,500
3,prod_T600_r1,pending,600
4,prod_T700_r1,pending,700
5,prod_T800_r1,pending,800
6,prod_T300_r2,pending,300
7,prod_T400_r2,pending,400
8,prod_T500_r2,pending,500
9,prod_T600_r2,pending,600


## 4. 待数据完成的讨论与结论

尚无完整正式数据，不给出最终系数、误差排名或“已收敛”判断。完成后按实际数据生成晶格常数曲线、温压稳定性、分块误差、三项参数验证及实验偏差。

## 参考文献与复现

完整参考文献和横向方法比较见[文献调研](docs/literature_review.md)，势文件说明见[来源](potentials/SOURCE.md)。运行步骤、目录树及未完成事项见[README](README.md)。Notebook只读取结果和展示图，不在打开时自动启动数小时的模拟。